In [2]:
import os
import json
import pandas as pd
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from concurrent.futures import ThreadPoolExecutor

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

In [2]:
pip install pyinaturalist

Note: you may need to restart the kernel to use updated packages.


In [3]:
import io
import os
import csv
import random
import pandas as pd
from PIL import Image, ImageEnhance
from concurrent.futures import ThreadPoolExecutor

# Scrape Relevant Photos from iNaturalist

In [4]:
import time
import requests
from pyinaturalist import get_observations

import csv
from datetime import datetime

In [5]:
ls

'!round1 testing'/                  charts/
'Gemini Confidence Testing.ipynb'   charts_all_images/
 _baseline_gemini_2.5_flash/        charts_one_photo_testing.ipynb
 _cdrl_gemini_2.5_flash/            dataset_metadata.csv
 _cove_gemini_2.5_flash/            distortion_images/
 _far_gemini_2.5_flash/             gemini_beached_analysis.csv
 _metadata_gemini_2.5_flash/        hurt_animal_dataset_images/
 _ssot_gemini_2.5_flash/            hurt_animal_dataset_images_filtered/
 animal_dataset_images/             hurt_dataset_metadata.csv


In [6]:
# want to scrape these species from iNaturalist
# add birds, beavers, etc

animal_scrape_dict = {
  'California Sea Lion' : 'Zalophus californianus',
  'Northern Elephant Seal': 'Mirounga angustirostris',
  'Harbor Seal' : 'Phoca vitulina',
  'Common Dolphin' : 'Delphinus delphis',
  'Harbour Porpoise' : 'Phocoena phocoena',
  'American Beaver' : 'Castor canadensis',
  'Loggerhead Sea Turtle' : 'Caretta caretta',
  'Eastern Gray Squirrel' : 'Sciurus carolinensis',
  'White-Tailed Deer' : 'Odocoileus virginianus',
  'House Finch' : 'Haemorhous mexicanus',
  'Sea Otter' : 'Enhydra lutris',
  'Coyote' : 'Canis latrans',
}

# animal_scrape_dict = {
#   'California Sea Lion' : 'Zalophus californianus',
#   'Northern Elephant Seal': 'Mirounga angustirostris',
#   'Harbor Seal' : 'Phoca vitulina',
#   'Common Dolphin' : 'Delphinus delphis',
#   'Harbour Porpoise' : 'Phocoena phocoena',
# }


In [7]:
IMAGE_FOLDER = "hurt_animal_dataset_images"
CSV_FILE_NAME = "hurt_dataset_metadata.csv"
os.makedirs(IMAGE_FOLDER, exist_ok=True)

In [7]:
SEARCH_TERMS = ['injur', 'blood', 'suffer', 'distress', 'pain', 'hurt', 'beached'] # Add more as needed
MAX_PHOTOS_PER_SPECIES = 50

os.makedirs(IMAGE_FOLDER, exist_ok=True)

fieldnames = [
    'animal_key', 'animal_value', 'observation_id', 
    'latitude', 'longitude', 'date', 'time', 
    'description', 'image_filename'
]

# Initialize and overwrite the CSV
with open(CSV_FILE_NAME, mode='w', newline='', encoding='utf-8') as main_csv:
    writer = csv.DictWriter(main_csv, fieldnames=fieldnames)
    writer.writeheader()

# Global tracker to avoid duplicates across different species or terms
# If an ID is in this set, we skip it entirely
downloaded_ids = set()

# --- Main Scrape Loop ---
for key, value in animal_scrape_dict.items():
    print(f"\n{'='*40}")
    print(f"Target Species: {value} ({key})")
    print(f"{'='*40}")
    
    photos_per_species_count = 0
    
    for term in SEARCH_TERMS:
        # Stop searching if we already hit the limit for this species
        if photos_per_species_count >= MAX_PHOTOS_PER_SPECIES:
            break
            
        print(f"Searching for term: '{term}'...")
        
        observations = get_observations(
            q=term,
            search_on='description',
            taxon_name=value,
            photos=True,
            quality_grade='research',
            per_page=200,
            order_by='created_at',
            order='desc',
            d1="2025-02-01", #knowledge cutoff for gemini 2.5 and 3
        )
    
        results = observations.get('results', [])
        
        # Open in append mode to save results incrementally
        with open(CSV_FILE_NAME, mode='a', newline='', encoding='utf-8') as main_csv:
            writer = csv.DictWriter(main_csv, fieldnames=fieldnames)
    
            for obs in results:
                # BREAK if we hit 200 for this specific animal
                if photos_per_species_count >= MAX_PHOTOS_PER_SPECIES:
                    break
                
                obs_id = obs['id']
                
                # AVOID DUPLICATES
                if obs_id in downloaded_ids:
                    continue
    
                # 1. Location Parsing
                location = obs.get('location')
                lat, lon = (None, None)
                if location:
                    if isinstance(location, list) and len(location) == 2:
                        lat, lon = location[0], location[1]
                    elif isinstance(location, str):
                        parts = location.split(',')
                        if len(parts) == 2:
                            lat, lon = parts[0], parts[1]
    
                # 2. Date & Time Parsing
                time_raw = obs.get('created_at_details')
                obs_date, obs_time = ("Unknown", "Unknown")
                if time_raw:
                    obs_date = time_raw.get('date', "Unknown")
                    obs_time = time_raw.get('hour', "Unknown")
    
                # 3. Description Cleaning
                desc = (obs.get('description') or "").replace('\n', ' ').replace('\r', ' ')
    
                # 4. Image Download
                file_name = f"{obs_id}.jpg"
                file_path = os.path.join(IMAGE_FOLDER, file_name)
                
                success = False
                if obs['photos']:
                    img_url = obs['photos'][0]['url'].replace('square', 'large')
                    try:
                        img_data = requests.get(img_url, timeout=10).content
                        with open(file_path, 'wb') as f:
                            f.write(img_data)
                        success = True
                    except Exception as e:
                        print(f"  Error downloading {obs_id}: {e}")
    
                # 5. Save and Increment
                if success:
                    writer.writerow({
                        'animal_key': key,
                        'animal_value': value,
                        'observation_id': obs_id,
                        'latitude': lat,
                        'longitude': lon,
                        'date': obs_date,
                        'time': obs_time,
                        'description': desc,
                        'image_filename': file_name
                    })
                    downloaded_ids.add(obs_id) # Mark as downloaded
                    photos_per_species_count += 1

    # Small cooldown between keyword searches for the same species
    time.sleep(2)

    # print(f"Finished {value}. Total images for this species: {photos_per_species_count}")
    # print("Waiting 15 seconds for API cooldown...")
    # time.sleep(15)

print("\nFull dataset complete!")


Target Species: Zalophus californianus (California Sea Lion)
Searching for term: 'injur'...
Searching for term: 'blood'...
Searching for term: 'suffer'...
Searching for term: 'distress'...
Searching for term: 'pain'...
Searching for term: 'hurt'...
Searching for term: 'beached'...

Target Species: Mirounga angustirostris (Northern Elephant Seal)
Searching for term: 'injur'...
Searching for term: 'blood'...
Searching for term: 'suffer'...
Searching for term: 'distress'...
Searching for term: 'pain'...
Searching for term: 'hurt'...
Searching for term: 'beached'...

Target Species: Phoca vitulina (Harbor Seal)
Searching for term: 'injur'...
Searching for term: 'blood'...
Searching for term: 'suffer'...
Searching for term: 'distress'...
Searching for term: 'pain'...
Searching for term: 'hurt'...
Searching for term: 'beached'...

Target Species: Delphinus delphis (Common Dolphin)
Searching for term: 'injur'...
Searching for term: 'blood'...
Searching for term: 'suffer'...
Searching for ter

In [8]:
ls

'!round1 testing'/                  charts/
'Gemini Confidence Testing.ipynb'   charts.ipynb
 _baseline_gemini_2.5_flash/        charts_all_images/
 _cdrl_gemini_2.5_flash/            dataset_metadata.csv
 _cove_gemini_2.5_flash/            distortion_images/
 _far_gemini_2.5_flash/             gemini_beached_analysis.csv
 _metadata_gemini_2.5_flash/        hurt_animal_dataset_images/
 _ssot_gemini_2.5_flash/            hurt_animal_dataset_images_filtered/
 animal_dataset_images/             hurt_dataset_metadata.csv


In [30]:
len(os.listdir('./hurt_animal_dataset_images_filtered/'))

151

In [28]:

# --- Configuration ---
PROJECT_ID = 'spring-2026-lifejacket'
LOCATION = 'us-central1'
IMAGE_FOLDER = "hurt_animal_dataset_images_filtered"
INPUT_CSV = "hurt_dataset_metadata.csv"
OUTPUT_CSV = "gemini_beached_analysis.csv"

vertexai.init(project=PROJECT_ID, location=LOCATION)

# Gemini confidence testing

## Confidence ranges for a given photo

In [31]:
# --- Configuration ---
PROJECT_ID = 'spring-2026-lifejacket'
LOCATION = 'us-central1'
IMAGE_FOLDER = "animal_dataset_images"
INPUT_CSV = "dataset_metadata.csv"
vertexai.init(project=PROJECT_ID, location=LOCATION)

In [32]:
os.getcwd()

'/home/jupyter/Gemini Confidence Testing'

In [12]:
#beached images

#dolphin: 361231332
#turtle: 362234822, 362502571

#  362491926
#362471929

In [13]:
## find one photo for each species, and give gemini the same prompt and photo. 
## see what the distribution of confidence scores are
## try color jittering and rotation to see if that does anything to confidence as well

#california sea lion: 	observation_id: 362499770
#harbor seal:	observation_id: 362345207
#turtle: observation_id: 362423278  
#common dolphin: 361231332
#canada goose: 362736713
#domestic dog: 362424973


In [14]:
#new: 362471929, 362424973, 362588731

In [33]:
MODEL = 'gemini-2.5-flash'
model = GenerativeModel(MODEL)

/opt/conda/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [34]:
TARGET_IDS = [361231332, 362234822, 362471929, 362424973, 362588731]

TRIALS_PER_LEVEL = 50  # 50 trials * 5 distortion levels = 100 API requests per ID
MAX_WORKERS = 15       # Concurrent threads hitting the API simultaneously

In [35]:
DISTORTION_CONFIGS = {
    # Level 0: Perfect baseline (100% legibility)
    0: {"brightness": (1.0, 1.0), "contrast": (1.0, 1.0), "color": (1.0, 1.0), "sharpness": (1.0, 1.0), "rotate": False},
    
    # Level 1: Clean but slightly off (Mild exposure shift, clear details remain)
    1: {"brightness": (0.80, 1.20), "contrast": (0.85, 1.15), "color": (0.75, 1.25), "sharpness": (0.70, 1.30), "rotate": False},
    
    # Level 2: Noticeably muddy & blurred (Forced structural soften, losing fine textures)
    2: {"brightness": (0.60, 0.79), "contrast": (0.65, 0.84), "color": (0.50, 0.74), "sharpness": (0.20, 0.50), "rotate": False},
    
    # Level 3: Severe contrast loss & rotation (Highly faded, washed out color, orientation flipped)
    3: {"brightness": (0.40, 0.59), "contrast": (0.35, 0.64), "color": (0.15, 0.49), "sharpness": (0.01, 0.15), "rotate": True},
    
    # Level 4: Extreme degradation (Heavy silhouette or extreme overexposure ghosting)
    4: {"brightness": (0.20, 0.39), "contrast": (0.20, 0.34), "color": (0.01, 0.14), "sharpness": (0.0, 0.0), "rotate": True},
    
    # Level 5: The "Ghostly Fog" Threshold (Fully visible details are erased into a flat, blurry gray haze)
    5: {"brightness": (0.45, 0.55), "contrast": (0.05, 0.12), "color": (0.0, 0.0), "sharpness": (0.0, 0.0), "rotate": True}
}
OUTPUT_FIELDS = [
    'trial', 'distortion_level', 'observation_id', 'actual_name', 'actual_species', 'technique_output',
    'top1_common_name', 'top1_genus', 'top1_species', 'top1_prob',
    'top2_common_name', 'top2_genus', 'top2_species', 'top2_prob', 'question'
]

In [36]:
def get_baseline_prompt(metadata_context):
    return f"""You are an expert wildlife species identification model.

TASK: Identify the top 2 likely species based on image.

RULES:
- Probabilities must be integers (0-100) and do not need to sum to 100.
- A high probability requires definitive, unambiguous visual traits. Be honest; avoid overconfidence.
- The Maximum Confidence Query must be a safe, non-invasive question an observer can answer using visual cues alone from a distance (e.g., using a camera zoom or waiting for a specific angle). Do not ask to measure, touch, or approach the animal.

OUTPUT FORMAT:
Your response must consist of exactly two lines of text: your baseline status followed immediately by pipe-separated data:
Baseline Status: [Acknowledge baseline status]
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability | Maximum Confidence Query

EXAMPLE:
Baseline Status: Baseline
Harbor Seal | Phoca | Phoca vitulina | 85 | Grey Seal | Halichoerus | Halichoerus grypus | 40 | Can you use a camera zoom to get a clear photo of the nostril shape from a direct front-facing angle?"""

In [37]:
def get_baseline_metadata_prompt(metadata_context):
    return f"""You are an expert wildlife species identification model.
OBSERVATION DATA: {metadata_context}

TASK: Identify the top 2 likely species based on image, date/time image was taken, and location of image.

RULES:
- Probabilities must be integers (0-100) and do not need to sum to 100.
- A high probability requires definitive, unambiguous visual traits. Be honest; avoid overconfidence.
- The Maximum Confidence Query must be a safe, non-invasive question an observer can answer using visual cues alone from a distance (e.g., using a camera zoom or waiting for a specific angle). Do not ask to measure, touch, or approach the animal.

OUTPUT FORMAT:
Your response must consist of exactly two lines of text: your baseline status followed immediately by pipe-separated data:
Baseline Status: [Acknowledge baseline status]
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability | Maximum Confidence Query

EXAMPLE:
Baseline Status: Baseline with location, date, time
Harbor Seal | Phoca | Phoca vitulina | 85 | Grey Seal | Halichoerus | Halichoerus grypus | 40 | Can you use a camera zoom to get a clear photo of the nostril shape from a direct front-facing angle?"""

In [38]:
def get_ssot_prompt(metadata_context):
    return f"""You are an expert wildlife species identification model executing a String Seed of Thought (SSoT) calibration process.
OBSERVATION DATA: {metadata_context}

TASK: Identify the top 2 likely species based on image, date/time image was taken, and location of image.

RULES:
- Probabilities must be integers (0-100) and do not need to sum to 100.
- A high probability requires definitive, unambiguous visual traits. Be honest; avoid overconfidence.
- The Maximum Confidence Query must be a safe, non-invasive question an observer can answer using visual cues alone from a distance (e.g., using a camera zoom or waiting for a specific angle). Do not ask to measure, touch, or approach the animal.
- You must generate a completely random 16-character alphanumeric string as the very first line of your response to anchor your internal confidence variance.

OUTPUT FORMAT:
Your response must consist of exactly two lines of text—the seed string followed immediately by the pipe-separated data:
Seed String: [16-character alphanumeric string]
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability | Maximum Confidence Query

EXAMPLE:
Seed String: aB8mXz2q11pL99Wv
Harbor Seal | Phoca | Phoca vitulina | 64 | Grey Seal | Halichoerus | Halichoerus grypus | 31 | Can you use a camera zoom to get a clear photo of the nostril shape from a direct front-facing angle?"""

In [39]:
def get_cdrl_prompt(metadata_context):
    return f"""You are an expert wildlife species identification model executing a CDRL calibration process.
OBSERVATION DATA: {metadata_context}

TASK: Identify the top 2 likely species based on image, date/time image was taken, and location of image.

RULES:
- Probabilities must be integers (0-100) and do not need to sum to 100.
- A high probability requires definitive, unambiguous visual traits. Be honest; avoid overconfidence.
- The Maximum Confidence Query must be a safe, non-invasive question an observer can answer using visual cues alone from a distance (e.g., using a camera zoom or waiting for a specific angle). Do not ask to measure, touch, or approach the animal.
- You must scale down probabilitie if your printed data gaps reveal critical missing features.

OUTPUT FORMAT:
Your response must consist of exactly two lines of text—your limitation analysis followed immediately by the pipe-separated data:
Gaps & Limits: [Identify what specific diagnostic features are missing from the raw data and evaluate their constraints]
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability | Maximum Confidence Query

EXAMPLE:
Gaps & Limits: Head profile and flank markings are completely obscured by waves; high look-alike overlap.
Harbor Seal | Phoca | Phoca vitulina | 55 | Grey Seal | Halichoerus | Halichoerus grypus | 40 | Can you use a camera zoom to get a clear photo of the nostril shape from a direct front-facing angle?"""

In [41]:
def get_far_prompt(metadata_context):
    return f"""You are an expert wildlife species identification model executing a Fact-and-Reflection (FaR) calibration process.
OBSERVATION DATA: {metadata_context}

TASK: Identify the top 2 likely species based on image, date/time image was taken, and location of image.

RULES:
- Probabilities must be integers (0-100) and do not need to sum to 100.
- A high probability requires definitive, unambiguous visual traits. Be honest; avoid overconfidence.
- The Maximum Confidence Query must be a safe, non-invasive question an observer can answer using visual cues alone from a distance (e.g., using a camera zoom or waiting for a specific angle). Do not ask to measure, touch, or approach the animal.
- Scale down probabilities if your reflection reveals close look-alike alternatives that match your listed facts.

OUTPUT FORMAT:
Your response must consist of exactly two lines of text—your factual reflection followed immediately by the pipe-separated data:
Facts & Reflection: [List objective visual traits present in the data and evaluate alternatives that match them]
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability | Maximum Confidence Query

EXAMPLE:
Facts & Reflection: Small dark spots, short tail, coastal rocks; muzzle profiles are missing, leaving candidate overlap.
Harbor Seal | Phoca | Phoca vitulina | 60 | Grey Seal | Halichoerus | Halichoerus grypus | 35 | Can you use a camera zoom to get a clear photo of the nostril shape from a direct front-facing angle?"""

In [42]:
def get_cove_prompt(metadata_context):
    return f"""You are an expert wildlife species identification model executing a Chain-of-Verification (CoVe) calibration process.
OBSERVATION DATA: {metadata_context}

TASK: Identify the top 2 likely species based on image, date/time image was taken, and location of image.

RULES:
- Probabilities must be integers (0-100) and do not need to sum to 100.
- A high probability requires definitive, unambiguous visual traits. Be honest; avoid overconfidence.
- The Maximum Confidence Query must be a safe, non-invasive question an observer can answer using visual cues alone from a distance (e.g., using a camera zoom or waiting for a specific angle). Do not ask to measure, touch, or approach the animal.
- Reduce probabilities if your cross-examination verification loop reveals unverified assumptions.

OUTPUT FORMAT:
Your response must consist of exactly two lines of text—your verification checks followed immediately by the pipe-separated data:
Verification Loop: [Formulate specific cross-examination questions and answer them using only explicit details]
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability | Maximum Confidence Query

EXAMPLE:
Verification Loop: Q: Is muzzle convex? A: No, flat. Q: Is geographic location in range? A: Yes, matches.
Harbor Seal | Phoca | Phoca vitulina | 85 | Grey Seal | Halichoerus | Halichoerus grypus | 20 | Can you use a camera zoom to get a clear photo of the nostril shape from a direct front-facing angle?"""

In [48]:
def get_best_prompt(metadata_context):
    return f"""You are an expert taxonomist executing a Diagnostic-Centric Calibration.
OBSERVATION DATA: {metadata_context}

TASK: Identify the top 2 likely species using a strict, evidence-based verification loop.

RULES:
1. DECOMPOSE: Isolate visible diagnostic markers (ear shape, coat, size).
2. REFLECT & AUDIT: Contrast your findings against the location, time, and photo itself, and known "look-alikes." Identify discrepancies.
3. CALIBRATE: Penalize probability if features are obscured or range is atypical.
4. MAXIMUM CONFIDENCE QUERY: Ask ONE non-invasive, visual question to confirm the #1 candidate.

PROBABILITY RULES:
- Probabilities must be integers (0-100) and do not need to sum to 100.
- A high probability requires definitive, unambiguous visual traits. Be honest; avoid overconfidence.

OUTPUT FORMAT:
Your response must consist of exactly two lines of text—your limitation analysis followed immediately by the pipe-separated data:
Gaps & Limits: [Identify what specific diagnostic features are missing from the raw data and evaluate their constraints]
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability | Maximum Confidence Query

EXAMPLE:
Gaps & Limits: Head profile and flank markings are completely obscured by waves; high look-alike overlap.
Harbor Seal | Phoca | Phoca vitulina | 55 | Grey Seal | Halichoerus | Halichoerus grypus | 40 | Can you get a clear photo of the nostril shape from a front-facing angle?"""

In [49]:

def process_single_row(task):
    trial_id, distortion_level, is_first_of_level, row, csv_filename, target_img_dir = task
    img_path = os.path.join(IMAGE_FOLDER, row['image_filename'])
    
    if not os.path.exists(img_path):
        return None

    metadata_context = (
        f"Location: ({row['latitude']}, {row['longitude']}), "
        f"Date/Time: {row['date']} at {row['time']} Military time"
    )

    try:
        cfg = DISTORTION_CONFIGS[distortion_level]
        
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            
            if cfg["rotate"]:
                rotation_choice = random.choice([Image.ROTATE_90, Image.ROTATE_180, Image.ROTATE_270])
                if rotation_choice:
                    img = img.transpose(rotation_choice)
            
            img = ImageEnhance.Brightness(img).enhance(random.uniform(*cfg["brightness"]))
            img = ImageEnhance.Contrast(img).enhance(random.uniform(*cfg["contrast"]))
            img = ImageEnhance.Color(img).enhance(random.uniform(*cfg["color"]))
            img = ImageEnhance.Sharpness(img).enhance(random.uniform(*cfg["sharpness"]))
            
            # Save transformed image to bytes for API delivery
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='JPEG', quality=85)
            image_bytes = img_byte_arr.getvalue()
            
            # === SAVES ONLY 1 SAMPLE PER DISTORTION LEVEL ===
            if is_first_of_level:
                label = "normal" if distortion_level == 0 else f"level_{distortion_level}"
                preview_filename = f"sample_{label}.jpg"
                preview_path = os.path.join(target_img_dir, preview_filename)
                img.save(preview_path, format='JPEG', quality=85)

        image_part = Part.from_data(data=image_bytes, mime_type="image/jpeg")

        #get diff prompts here:
        prompt = get_best_prompt(metadata_context=metadata_context)
        #prompt = get_ssot_prompt(metadata_context=metadata_context)
        #prompt = get_cdrl_prompt(metadata_context=metadata_context)
        #prompt = get_far_prompt(metadata_context=metadata_context)
        #prompt = get_cove_prompt(metadata_context=metadata_context)
        
        response = model.generate_content([prompt, image_part])
        raw_res = response.text.strip()
    except Exception as e:
        # Appended 'Error' to match the new 9-column shape
        raw_res = "Error | Error | Error | 0 | Error | Error | Error | 0 | Error"

    # 1. Split raw response into lines and handle empty string edge cases safely
    lines = [line.strip() for line in raw_res.strip().split('\n') if line.strip()]
    
    if len(lines) > 1:
        # Expected case: Line 1 is the technique metadata, Line 2 is the data row
        technique_line = lines[0]
        data_line = lines[1]
    else:
        # Fallback/Error case: if only one line came back, treat it as the data row
        technique_line = "N/A"
        data_line = lines[0] if lines else "Error | Error | Error | 0 | Error | Error | Error | 0 | Error"

    # 2. Split the data line by the pipe character
    parts = [p.strip() for p in data_line.split('|')]
    
    # Pad to 9 columns to account for the new query field safely
    while len(parts) < 9:
        parts.append("N/A")

    result = {
        'trial': trial_id,
        'distortion_level': distortion_level,
        'observation_id': row['observation_id'],
        'actual_name': row['animal_key'],
        'actual_species': row['animal_value'],
        'technique_output': technique_line,  # Stores Line 1 (Seeds, Facts, Gaps, or Verifications)
        'top1_common_name': parts[0],
        'top1_genus': parts[1],
        'top1_species': parts[2],
        'top1_prob': parts[3],
        'top2_common_name': parts[4],
        'top2_genus': parts[5],
        'top2_species': parts[6],
        'top2_prob': parts[7],
        'question': parts[8], 
    }

    # Thread-safe appending directly to CSV
    with open(csv_filename, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
        writer.writerow(result)

    return result



In [50]:
INPUT_CSV

'dataset_metadata.csv'

In [51]:
df = pd.read_csv(INPUT_CSV)

In [52]:
matching_rows = df[df['observation_id'] == 362234822]
matching_rows

,animal_key,animal_value,observation_id,latitude,longitude,date,time,description,image_filename
303,Loggerhead Sea Turtle,Caretta caretta,362234822,19.704331,-156.177252,2026-05-16,17,NaN,362234822.jpg


In [53]:
df = pd.read_csv(INPUT_CSV)

BASE_PREVIEW_DIR = "distortion_images"
os.makedirs(BASE_PREVIEW_DIR, exist_ok=True)

for idx, obs_id in enumerate(TARGET_IDS, start=1):
    print(f"--- [Batch {idx}/{len(TARGET_IDS)}] Profiling Observation ID: {obs_id} ---")
    
    matching_rows = df[df['observation_id'] == obs_id]
    if matching_rows.empty:
        print(f"observation_id {obs_id} not indexed. Skipping.")
        continue
        
    target_row = matching_rows.iloc[0]
    
    dynamic_output_csv = f"{obs_id}_distribution_results.csv"
    target_img_dir = os.path.join(BASE_PREVIEW_DIR, f"images_{obs_id}")
    os.makedirs(target_img_dir, exist_ok=True)
    
    with open(dynamic_output_csv, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
        writer.writeheader()
        
    # Compile parameter loop spanning Level 0 through Level 5 (6 loops total)
    tasks = []
    global_trial_count = 1
    for level in range(0, 6):
        for inner_idx in range(TRIALS_PER_LEVEL):
            # If inner_idx is 0, this is the very first trial running for this specific level tier
            is_first_of_level = (inner_idx == 0)
            tasks.append((global_trial_count, level, is_first_of_level, target_row, dynamic_output_csv, target_img_dir))
            global_trial_count += 1
            
    print(f"Dispatching {len(tasks)} parallel requests (Isolated to 6 reference images saved)...")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        _ = list(executor.map(process_single_row, tasks))
        
    print(f"Complete. Output CSV: {dynamic_output_csv}")
    print(f"Saved 6 reference image files inside: {target_img_dir}\n")

print("All batch profiles executed. System processing run fully finished.")

--- [Batch 1/5] Profiling Observation ID: 361231332 ---
Dispatching 300 parallel requests (Isolated to 6 reference images saved)...
Complete. Output CSV: 361231332_distribution_results.csv
Saved 6 reference image files inside: distortion_images/images_361231332

--- [Batch 2/5] Profiling Observation ID: 362234822 ---
Dispatching 300 parallel requests (Isolated to 6 reference images saved)...
Complete. Output CSV: 362234822_distribution_results.csv
Saved 6 reference image files inside: distortion_images/images_362234822

--- [Batch 3/5] Profiling Observation ID: 362471929 ---
Dispatching 300 parallel requests (Isolated to 6 reference images saved)...
Complete. Output CSV: 362471929_distribution_results.csv
Saved 6 reference image files inside: distortion_images/images_362471929

--- [Batch 4/5] Profiling Observation ID: 362424973 ---
Dispatching 300 parallel requests (Isolated to 6 reference images saved)...
Complete. Output CSV: 362424973_distribution_results.csv
Saved 6 reference imag

# All Images

In [34]:
DISTORTION_CONFIGS = {
    # Level 0: Perfect baseline (100% legibility)
    0: {"brightness": (1.0, 1.0), "contrast": (1.0, 1.0), "color": (1.0, 1.0), "sharpness": (1.0, 1.0), "rotate": False},
    
    # Level 1: Clean but slightly off (Mild exposure shift, clear details remain)
    1: {"brightness": (0.80, 1.20), "contrast": (0.85, 1.15), "color": (0.75, 1.25), "sharpness": (0.70, 1.30), "rotate": False},
    
    # Level 2: Noticeably muddy & blurred (Forced structural soften, losing fine textures)
    2: {"brightness": (0.60, 0.79), "contrast": (0.65, 0.84), "color": (0.50, 0.74), "sharpness": (0.20, 0.50), "rotate": False},
    
    # Level 3: Severe contrast loss & rotation (Highly faded, washed out color, orientation flipped)
    3: {"brightness": (0.40, 0.59), "contrast": (0.35, 0.64), "color": (0.15, 0.49), "sharpness": (0.01, 0.15), "rotate": True},
    
    # Level 4: Extreme degradation (Heavy silhouette or extreme overexposure ghosting)
    4: {"brightness": (0.20, 0.39), "contrast": (0.20, 0.34), "color": (0.01, 0.14), "sharpness": (0.0, 0.0), "rotate": True},
    
    # Level 5: The "Ghostly Fog" Threshold (Fully visible details are erased into a flat, blurry gray haze)
    5: {"brightness": (0.45, 0.55), "contrast": (0.05, 0.12), "color": (0.0, 0.0), "sharpness": (0.0, 0.0), "rotate": True}
}
OUTPUT_FIELDS = [
    'trial', 'distortion_level', 'observation_id', 'actual_name', 'actual_species', 'technique_output',
    'top1_common_name', 'top1_genus', 'top1_species', 'top1_prob',
    'top2_common_name', 'top2_genus', 'top2_species', 'top2_prob', 'question'
]

In [35]:
def process_single_row(task):
    trial_id, distortion_level, row = task
    img_path = os.path.join(IMAGE_FOLDER, row['image_filename'])
    
    if not os.path.exists(img_path):
        return None

    metadata_context = (
        f"Location: ({row['latitude']}, {row['longitude']}), "
        f"Date/Time: {row['date']} at {row['time']} Military time"
    )

    try:
        cfg = DISTORTION_CONFIGS[distortion_level]
        
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            
            if cfg["rotate"]:
                rotation_choice = random.choice([Image.ROTATE_90, Image.ROTATE_180, Image.ROTATE_270])
                img = img.transpose(rotation_choice)
            
            img = ImageEnhance.Brightness(img).enhance(random.uniform(*cfg["brightness"]))
            img = ImageEnhance.Contrast(img).enhance(random.uniform(*cfg["contrast"]))
            img = ImageEnhance.Color(img).enhance(random.uniform(*cfg["color"]))
            img = ImageEnhance.Sharpness(img).enhance(random.uniform(*cfg["sharpness"]))
            
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='JPEG', quality=85)
            image_bytes = img_byte_arr.getvalue()

        image_part = Part.from_data(data=image_bytes, mime_type="image/jpeg")
        
        prompt = get_far_prompt(metadata_context=metadata_context)
        #print(prompt)
        
        response = model.generate_content([prompt, image_part])
        raw_res = response.text.strip()
    except Exception:
        raw_res = "Error | Error | Error | 0 | Error | Error | Error | 0"
        traceback.print_exc()

    # 1. Split raw response into lines and handle empty string edge cases safely
    lines = [line.strip() for line in raw_res.strip().split('\n') if line.strip()]
    
    if len(lines) > 1:
        # Expected case: Line 1 is the technique metadata, Line 2 is the data row
        technique_line = lines[0]
        data_line = lines[1]
    else:
        # Fallback/Error case: if only one line came back, treat it as the data row
        technique_line = "N/A"
        data_line = lines[0] if lines else "Error | Error | Error | 0 | Error | Error | Error | 0 | Error"

    # 2. Split the data line by the pipe character
    parts = [p.strip() for p in data_line.split('|')]
    
    # Pad to 9 columns to account for the new query field safely
    while len(parts) < 9:
        parts.append("N/A")

    result = {
        'trial': trial_id,
        'distortion_level': distortion_level,
        'observation_id': row['observation_id'],
        'actual_name': row['animal_key'],
        'actual_species': row['animal_value'],
        'technique_output': technique_line,  # Stores Line 1 (Seeds, Facts, Gaps, or Verifications)
        'top1_common_name': parts[0],
        'top1_genus': parts[1],
        'top1_species': parts[2],
        'top1_prob': parts[3],
        'top2_common_name': parts[4],
        'top2_genus': parts[5],
        'top2_species': parts[6],
        'top2_prob': parts[7],
        'question': parts[8], 
    }

    # Thread-safe appending to the single output file
    with open(OUTPUT_CSV, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
        writer.writerow(result)

    return result


In [36]:
df = pd.read_csv(INPUT_CSV)

In [37]:
df.head()

,animal_key,animal_value,observation_id,latitude,longitude,date,time,description,image_filename
0,California Sea Lion,Zalophus californianus,364110724,34.003536,-118.804019,2026-05-22,15,Injured and/or ill. Called the Marine Mammal R...,364110724.jpg
1,California Sea Lion,Zalophus californianus,363244761,36.855400,-121.813618,2026-05-19,20,sea lion pup with injured back? couldn't stage...,363244761.jpg
2,California Sea Lion,Zalophus californianus,358525456,32.850389,-117.272943,2026-05-04,22,injured - garbage stuck around neck,358525456.jpg
3,California Sea Lion,Zalophus californianus,354787599,36.602951,-121.892296,2026-04-26,15,This sea lion was injured. There was a fishing...,354787599.jpg
4,California Sea Lion,Zalophus californianus,343620501,37.810935,-122.411337,2026-03-18,4,"At the the san Francisco pier, very loud, one ...",343620501.jpg


In [38]:
OUTPUT_CSV = 'all_images_distortion_results.csv'
TRIALS_PER_LEVEL=2
MAX_WORKERS=15
IMAGE_FOLDER = "hurt_animal_dataset_images_filtered"

In [39]:
INPUT_CSV

'hurt_dataset_metadata.csv'

In [40]:


with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
    writer.writeheader()

df = pd.read_csv(INPUT_CSV)
tasks = []
global_trial_id = 1

print(f"Preparing tasks for {len(df)} images...")

# 2. Build task list: Level 0-5 for every single row in the CSV
for _, row in df.iterrows():
    for level in range(0, 6):
        for _ in range(TRIALS_PER_LEVEL):
            tasks.append((global_trial_id, level, row))
            global_trial_id += 1

print(f"Starting processing: {len(tasks)} total requests...")

# 3. Process in parallel
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    _ = list(executor.map(process_single_row, tasks))

print(f"All processing finished. Results saved to: {OUTPUT_CSV}")


Preparing tasks for 398 images...
Starting processing: 4776 total requests...
All processing finished. Results saved to: all_images_distortion_results.csv
